# 03 — Hyperparameter Tuning & Final Evaluation

**Author:** Rudolph Otoo  
**Date:** 2026  

---

## Objective

Refine every model (except the hyperparameter-free dummy baseline) via
stratified 5-fold cross-validation on the **training split**. The validation
split serves as an independent hold-out during tuning; the **held-out test set**
is used only once at the end.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.config import Paths, Settings
from src.data import process_data
from src.features import split_features_target
from src.models import build_pipeline, list_models
from src.tune import tune_model
from src.evaluate import evaluate_model, aggregate_benchmark_tables
from src.visualize import (
    set_global_style,
    plot_roc_curves,
    plot_confusion_matrices,
    plot_feature_importance,
)

set_global_style()
paths = Paths()

## 1. Data Loading & Splitting

Identical to notebook 02; uses the same deterministic seed to produce
exactly the same train/val/test membership.

In [ ]:
settings = Settings()
frame    = process_data()
splits   = split_features_target(frame, settings=settings)

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

## 2. Hyperparameter Tuning (Stratified 5-Fold CV)

Each model's search space is defined in `src/models.py:get_param_grid()`.
The best hyperparameters are selected by maximising ROC-AUC across folds.
The dummy classifier is fit directly (no grid search needed).

In [ ]:
tuned_pipelines = {}
best_params     = {}

for model_name in list_models():
    fitted, params = tune_model(model_name, X_train, y_train, settings=settings)
    tuned_pipelines[model_name] = fitted
    best_params[model_name]     = params

pd.DataFrame(best_params).T

## 3. Validation Performance (Tuned Models)

Evaluate the tuned models on the validation set. This provides a first
signal of whether tuning improved generalisation relative to the untuned
defaults reported in notebook 02.

In [ ]:
val_results = {}
for name, pipe in tuned_pipelines.items():
    val_results[name] = evaluate_model(
        pipe, X_val, y_val, model_name=f"{name} (val)"
    )

aggregate_benchmark_tables(val_results)

## 4. Final Evaluation on Held-out Test Set

This is the **one and only** evaluation of the tuned models on the held-out
test set. Any conclusions drawn from this block are reported in the
manuscript / README.

In [ ]:
test_results = {}
for name, pipe in tuned_pipelines.items():
    test_results[name] = evaluate_model(
        pipe, X_test, y_test, model_name=name
    )

final_table = aggregate_benchmark_tables(test_results)
print("\n── Final benchmark (held-out test set) ──")
final_table

## 5. ROC Curves, Confusion Matrices & Feature Importance

In [ ]:
plot_roc_curves(tuned_pipelines, X_test, y_test)
plot_confusion_matrices(tuned_pipelines, X_test, y_test)

In [ ]:
plot_feature_importance(tuned_pipelines["gradient_boosting"])

## 6. Persist Best Model

Serialise the tuned Gradient Boosting pipeline for downstream deployment or
downstream analysis. The fitted `StandardScaler` is embedded within the
pipeline, so no separate scaler object is needed at inference time.

In [ ]:
best_model = tuned_pipelines["gradient_boosting"]
joblib.dump(best_model, paths.models / "best_gradient_boosting_pipeline.joblib")
print(f"Model saved to {paths.models / 'best_gradient_boosting_pipeline.joblib'}")

## 7. Key Conclusions

| Model | ROC-AUC (test) | F1-score (test) | Notes |
|---|---|---|---|
| Dummy | 0.50 | 0.00 | No discriminative ability (majority-class baseline) |
| Logistic Regression (L2) | ~0.80 | ~0.57 | Strong interpretable baseline; consistent with clinical literature |
| Random Forest | ~0.82 | ~0.57 | Marginal gain over logistic regression |
| **Gradient Boosting** | **~0.82** | **~0.59** | Best overall; selected for deployment |

*Numbers above reflect a single fixed-seed reference run. Recomputed values
may vary by a few hundredths since the tree ensembles are stochastic.*

**Gradients of improvement:**
- Dummy → LogisticRegression: large leap (~30 pp ROC-AUC); validates that features contain genuine signal.
- LogisticRegression → GradientBoosting: modest ~2 pp ROC-AUC gain; consistent with literature for tabular clinical data.
- All models benefit from median-imputation of structurally missing values (a deliberate choice documented in `src/data.py`).